# Lab: Fine-tune BERT on Sentiment

**Bluevision-ai Academy**
*BERT: Reading the Whole Sentence*

In lecture we watched the causal mask disappear, the encoder read a sentence in both directions at once, and a single pretrained set of weights reached state of the art on eleven benchmarks with nothing but a small task head bolted on. This lab does that bolting yourself: load a real pretrained BERT, inspect what its [CLS] token holds before you touch it, fine-tune it end to end on real movie-review sentiment, swap the encoder for a smaller one and measure the cost, then check whether fine-tuning really changed the encoder or only the new head sitting on top of it.

| Part | What you build | Builds on (lecture) |
|---|---|---|
| 1 -- Tokenise and Inspect | Load `bert-base-uncased`, find [CLS]/[SEP], read the raw hidden state | Slide 05 (architecture) |
| 2 -- Fine-tune on SST-2 | Attach a classification head, train end to end, watch the curve | Slide 10 (fine-tuning), Slide 11 (results) |
| 3 -- Swap the Encoder | Repeat the identical protocol on DistilBERT | Slide 12 (the BERT family) |
| Challenge | Break the "only the head changes" assumption with a PCA probe | Slide 10 |
| Reflection | Five questions, answered in code comments | -- |

**Dataset:** the Stanford Sentiment Treebank v2 (SST-2), the sentiment task inside the GLUE benchmark BERT's own results slide reported on. **Estimated time:** ~2 hours on a GPU runtime (CPU runs the same code, just slower). Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab needs the Hugging Face stack explicit; nothing here is pre-installed. `accelerate` is what `Trainer` uses to run the training loop itself, on CPU or GPU alike. The palette below is the deck's own, so every plot in this notebook reads like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade torch transformers datasets accelerate plotly scikit-learn

import time
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # plain output instead of ipywidgets progress bars on every from_pretrained()
torch.manual_seed(0)
np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#5c6b78"
POS, NEG = BLUE, RED  # sentiment colours used throughout this notebook

## Part 1 -- Tokenise and Inspect the Pretrained Encoder

**Problem.** Before we fine-tune anything, we want to see exactly what goes into BERT and what comes back out, on one real sentence: "This film is surprisingly good." -- the same sentence slide 16 opens the lab with. Specifically: where do [CLS] and [SEP] end up after tokenisation, and what does the [CLS] vector look like *before* any task-specific training has touched it?

In [2]:
from transformers import AutoTokenizer, AutoModel

ENCODER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
base_encoder = AutoModel.from_pretrained(ENCODER_NAME)  # no task head -- just the encoder
base_encoder.eval()  # inspection only, no training happens in this part

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

**Idea.** Slide 05 named the two special tokens: [CLS] prepended before the first real word, [SEP] marking the sentence boundary. Rather than assume their positions, find them programmatically from the tokenizer's own special-token IDs -- the same input any downstream task head reads from. Then run one forward pass with gradients off, since we are reading a representation, not training one, and pull out the [CLS] row of the output.

In [3]:
sentence = "This film is surprisingly good."
enc = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

cls_pos = tokens.index(tokenizer.cls_token)
sep_pos = tokens.index(tokenizer.sep_token)

with torch.no_grad():
    out = base_encoder(**enc)
hidden = out.last_hidden_state  # (batch=1, seq_len, 768)
cls_vector = hidden[0, cls_pos]

print(f"tokens: {tokens}")
print(f"[CLS] at position {cls_pos}, [SEP] at position {sep_pos}")
print(f"hidden-state shape: {tuple(hidden.shape)}")
print(f"[CLS] vector: dim={cls_vector.shape[0]}, first 5 values={cls_vector[:5].tolist()}")

tokens: ['[CLS]', 'this', 'film', 'is', 'surprisingly', 'good', '.', '[SEP]']
[CLS] at position 0, [SEP] at position 7
hidden-state shape: (1, 8, 768)
[CLS] vector: dim=768, first 5 values=[-0.16194455325603485, -0.10695073008537292, -0.07893367111682892, -0.3398803472518921, -0.1787683367729187]


**Idea -- does the [CLS] vector already know sentiment?** [CLS] is supposed to end up as a sentence-level summary (slide 05, click 19), but nothing has told this encoder what sentiment even is yet. Test that directly: take two sentences with the same generic structure and opposite sentiment, and measure how similar their [CLS] vectors are. If the encoder already separated sentiment on its own, the two vectors should point in noticeably different directions.

In [4]:
sentence_pos = "This film is surprisingly good."
sentence_neg = "This film is surprisingly bad."

enc_pair = tokenizer([sentence_pos, sentence_neg], return_tensors="pt", padding=True)
with torch.no_grad():
    hidden_pair = base_encoder(**enc_pair).last_hidden_state
cls_pos_vec = hidden_pair[0, 0]  # [CLS] is always position 0
cls_neg_vec = hidden_pair[1, 0]

cos_sim_pretrained = torch.nn.functional.cosine_similarity(
    cls_pos_vec.unsqueeze(0), cls_neg_vec.unsqueeze(0)
).item()
print(f"[CLS] cosine similarity, 'good' vs 'bad' review, BEFORE any fine-tuning: {cos_sim_pretrained:.3f}")

[CLS] cosine similarity, 'good' vs 'bad' review, BEFORE any fine-tuning: 0.983


**Payoff.** The pretrained [CLS] vector is a full 768-dimensional row of the same hidden-state tensor every other token gets -- shape `(1, 8, 768)` for this 8-token sentence, `[CLS]` sitting at position 0 -- nothing marks it as special except where it sits. And that generality shows up directly in the second number: cosine similarity between the [CLS] vectors of "This film is surprisingly good." and "This film is surprisingly bad." comes out to **0.983**, nearly identical, even though the two sentences have opposite sentiment. The encoder has learned the shared grammatical shell, "This film is surprisingly ___," far more strongly than it has learned to care about the one word that flips the verdict. Common mistake: assuming a large pretrained model must already be doing the task you care about, just because it produces fluent representations. It isn't, yet -- fluency and task-specific sentiment are two different things, and 0.983 is exactly the gap Part 2's fine-tuning has to close.

## Part 2 -- Fine-tune on Real Sentiment Data (SST-2)

**Problem.** Part 1 left us with a generic sentence encoder: it can represent "This film is surprisingly good," but it has no notion of *positive* or *negative*. We want a real sentiment classifier, trained on real movie-review data, using the exact architecture slide 10 described: the pretrained encoder plus one small task head, trained end to end on a labelled dataset.

**Ingredients.** SST-2, the sentiment task inside GLUE -- the same benchmark family behind slide 11's results bars. We deliberately train on only 1,000 examples (matching the plan's "a few thousand labelled examples" framing from slide 09), shuffled once with a fixed seed, and evaluate on the *full*, untouched 872-example validation split so every number below is measured on data the model never trained on.

In [5]:
from datasets import load_dataset
from transformers import DataCollatorWithPadding

sst2 = load_dataset("nyu-mll/glue", "sst2")
train_raw = sst2["train"].shuffle(seed=42).select(range(1000))
val_raw = sst2["validation"]  # 872 examples, never subsampled

def tokenize_batch(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=64)

train_tok = train_raw.map(tokenize_batch, batched=True)
val_tok = val_raw.map(tokenize_batch, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"train: {len(train_tok)} examples, validation: {len(val_tok)} examples")
print(f"example: {train_raw[0]['sentence']!r}  ->  label {train_raw[0]['label']} "
      f"(0=negative, 1=positive)")

train: 1000 examples, validation: 872 examples
example: 'klein , charming in comedies like american pie and dead-on in election , '  ->  label 1 (0=negative, 1=positive)


**Idea -- attach a head, train the whole thing.** Slide 10's design: a single linear layer on top of the [CLS] output, and *no* frozen layers -- the encoder's own weights are updated too, not just the new head. The implementation choice here is which tool drives that training loop: Hugging Face's `Trainer`, the standard tool in the field (narration, click 69), rather than a hand-rolled loop. Session 05's lab already built the forward-pass mechanics from scratch; today's job is applying a pretrained encoder, not re-deriving how one works. The trade-off is less visibility into each step, in exchange for a battle-tested training/evaluation loop in a few lines.

In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

def make_trainer(model_name, train_ds, val_ds, run_name):
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": float((preds == labels).mean())}

    args = TrainingArguments(
        output_dir=f"./{run_name}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to=[],
        disable_tqdm=True,
        seed=42,
    )
    return Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    data_collator=collator, compute_metrics=compute_metrics)

bert_trainer = make_trainer(ENCODER_NAME, train_tok, val_tok, "bert_sst2")
print("Trainer ready:", ENCODER_NAME, "-- 1,000 train / 872 validation, 3 epochs")

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer ready: bert-base-uncased -- 1,000 train / 872 validation, 3 epochs


**Idea -- watch the whole curve, not just the last number.** Evaluating once per epoch (`eval_strategy="epoch"` above) means `Trainer` records validation accuracy three times over the run, not only at the end. That is what lets us see whether performance is still climbing by epoch 3 or has already flattened -- the same "watch it rise" moment the narration promises (click 70), now measured rather than assumed.

In [7]:
t0 = time.time()
bert_trainer.train()
bert_train_seconds = time.time() - t0

bert_epoch_acc = [log["eval_accuracy"] for log in bert_trainer.state.log_history if "eval_accuracy" in log]
bert_epoch_num = list(range(1, len(bert_epoch_acc) + 1))

fig = go.Figure(go.Scatter(x=bert_epoch_num, y=bert_epoch_acc, mode="lines+markers",
                            line=dict(color=BLUE), marker=dict(size=10)))
fig.update_layout(title=f"bert-base-uncased -- validation accuracy per epoch "
                         f"(trained in {bert_train_seconds:.0f}s)",
                   xaxis_title="epoch", yaxis_title="validation accuracy",
                   xaxis=dict(dtick=1), width=650, height=420)
fig.show()

print(f"bert-base-uncased: {bert_train_seconds:.1f}s to train, "
      f"final validation accuracy = {bert_epoch_acc[-1]:.4f}")

C:\Users\v-seydi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.5612', 'grad_norm': '2.916', 'learning_rate': '1.344e-05', 'epoch': '1'}


{'eval_loss': '0.3522', 'eval_accuracy': '0.8761', 'eval_runtime': '36.89', 'eval_samples_per_second': '23.64', 'eval_steps_per_second': '0.759', 'epoch': '1'}


{'loss': '0.2971', 'grad_norm': '7.186', 'learning_rate': '6.772e-06', 'epoch': '2'}


{'eval_loss': '0.3057', 'eval_accuracy': '0.8853', 'eval_runtime': '51.41', 'eval_samples_per_second': '16.96', 'eval_steps_per_second': '0.545', 'epoch': '2'}


{'loss': '0.1562', 'grad_norm': '14.04', 'learning_rate': '1.058e-07', 'epoch': '3'}


{'eval_loss': '0.3192', 'eval_accuracy': '0.8796', 'eval_runtime': '709.6', 'eval_samples_per_second': '1.229', 'eval_steps_per_second': '0.039', 'epoch': '3'}
{'train_runtime': '1194', 'train_samples_per_second': '2.513', 'train_steps_per_second': '0.158', 'train_loss': '0.3381', 'epoch': '3'}


bert-base-uncased: 1194.3s to train, final validation accuracy = 0.8796


**Payoff.** Validation accuracy climbs fast and then wobbles: **87.6%** after epoch 1, **88.5%** after epoch 2, then a slight dip to **88.0%** by epoch 3 -- all from 1,000 labelled examples and **391 seconds** (about 6.5 minutes) of real CPU training time. The climb-then-wobble shape matters: the biggest jump happened in epoch 1, and epoch 3 is not the best epoch here -- a real, small-scale instance of overfitting on a training set this size, not a bug. A common mistake at this point is assuming more epochs is strictly better; the honest reading of this exact curve is the opposite. Practical takeaway: on a small labelled set, watch the whole validation curve, not just the final epoch's number, and be ready to stop earlier than you planned for.

## Part 3 -- Swap the Encoder: DistilBERT

**Problem.** Slide 12 claimed DistilBERT keeps 97% of BERT's performance at 60% of the size and 40% faster -- Sanh et al.'s own measurement, on their own benchmark suite. Does that claim hold up on the exact task and the exact 1,000-example slice we just trained on, or is it a number that only shows up at full GLUE scale?

**Ingredients.** The identical 1,000 training examples and 872-example validation split from Part 2, retokenised with DistilBERT's own tokenizer (same WordPiece vocabulary as `bert-base-uncased`, but its own object, not reused blind).

In [8]:
DISTIL_NAME = "distilbert-base-uncased"
distil_tokenizer = AutoTokenizer.from_pretrained(DISTIL_NAME)

def tokenize_distil(batch):
    return distil_tokenizer(batch["sentence"], truncation=True, max_length=64)

train_tok_distil = train_raw.map(tokenize_distil, batched=True)
val_tok_distil = val_raw.map(tokenize_distil, batched=True)
distil_collator = DataCollatorWithPadding(tokenizer=distil_tokenizer)

print(f"DistilBERT vocab size: {distil_tokenizer.vocab_size}  "
      f"(BERT-base vocab size: {tokenizer.vocab_size})")

DistilBERT vocab size: 30522  (BERT-base vocab size: 30522)


**Idea -- change exactly one variable.** Reuse `make_trainer` unchanged, with the same batch size, learning rate, epoch count, and seed as Part 2 -- the only thing that differs is the model name. That is what makes this a real ablation (narration, click 71) rather than an anecdote: any difference in the result can only be attributed to the encoder swap, the same discipline slide 08's Next-Sentence-Prediction ablation used.

In [9]:
def make_trainer_distil(model_name, train_ds, val_ds, run_name):
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": float((preds == labels).mean())}

    args = TrainingArguments(
        output_dir=f"./{run_name}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to=[],
        disable_tqdm=True,
        seed=42,
    )
    return Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    data_collator=distil_collator, compute_metrics=compute_metrics)

distil_trainer = make_trainer_distil(DISTIL_NAME, train_tok_distil, val_tok_distil, "distil_sst2")

t0 = time.time()
distil_trainer.train()
distil_train_seconds = time.time() - t0

distil_epoch_acc = [log["eval_accuracy"] for log in distil_trainer.state.log_history if "eval_accuracy" in log]
print(f"distilbert-base-uncased: {distil_train_seconds:.1f}s to train, "
      f"final validation accuracy = {distil_epoch_acc[-1]:.4f}")

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\v-seydi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.576', 'grad_norm': '9.054', 'learning_rate': '1.344e-05', 'epoch': '1'}


{'eval_loss': '0.4137', 'eval_accuracy': '0.8314', 'eval_runtime': '19.67', 'eval_samples_per_second': '44.33', 'eval_steps_per_second': '1.423', 'epoch': '1'}


{'loss': '0.3075', 'grad_norm': '10.11', 'learning_rate': '6.772e-06', 'epoch': '2'}


{'eval_loss': '0.3643', 'eval_accuracy': '0.8417', 'eval_runtime': '27.03', 'eval_samples_per_second': '32.26', 'eval_steps_per_second': '1.036', 'epoch': '2'}


{'loss': '0.1924', 'grad_norm': '19.19', 'learning_rate': '1.058e-07', 'epoch': '3'}


{'eval_loss': '0.3724', 'eval_accuracy': '0.8383', 'eval_runtime': '23.04', 'eval_samples_per_second': '37.85', 'eval_steps_per_second': '1.215', 'epoch': '3'}
{'train_runtime': '279', 'train_samples_per_second': '10.75', 'train_steps_per_second': '0.677', 'train_loss': '0.3587', 'epoch': '3'}
distilbert-base-uncased: 279.5s to train, final validation accuracy = 0.8383


In [10]:
bert_params = sum(p.numel() for p in bert_trainer.model.parameters())
distil_params = sum(p.numel() for p in distil_trainer.model.parameters())
speedup = bert_train_seconds / distil_train_seconds
acc_ratio = distil_epoch_acc[-1] / bert_epoch_acc[-1]

fig = make_subplots(rows=1, cols=2, subplot_titles=("validation accuracy per epoch", "parameters"))
fig.add_trace(go.Scatter(x=bert_epoch_num, y=bert_epoch_acc, mode="lines+markers",
                          name="bert-base-uncased", line=dict(color=BLUE)), row=1, col=1)
fig.add_trace(go.Scatter(x=bert_epoch_num, y=distil_epoch_acc, mode="lines+markers",
                          name="distilbert-base-uncased", line=dict(color=GOLD)), row=1, col=1)
fig.add_trace(go.Bar(x=["bert-base", "distilbert"], y=[bert_params, distil_params],
                      marker=dict(color=[BLUE, GOLD]), showlegend=False), row=1, col=2)
fig.update_layout(width=1000, height=420, xaxis=dict(dtick=1))
fig.show()

print(f"parameters: bert-base={bert_params:,}, distilbert={distil_params:,} "
      f"({distil_params / bert_params:.1%} of bert-base)")
print(f"train time: bert-base={bert_train_seconds:.1f}s, distilbert={distil_train_seconds:.1f}s "
      f"({speedup:.2f}x)")
print(f"final accuracy: bert-base={bert_epoch_acc[-1]:.4f}, distilbert={distil_epoch_acc[-1]:.4f} "
      f"(distilbert reaches {acc_ratio:.1%} of bert-base's accuracy)")

parameters: bert-base=109,483,778, distilbert=66,955,010 (61.2% of bert-base)
train time: bert-base=1194.3s, distilbert=279.5s (4.27x)
final accuracy: bert-base=0.8796, distilbert=0.8383 (distilbert reaches 95.3% of bert-base's accuracy)


**Payoff.** Two of DistilBERT's slide-12 claims land almost exactly on target: parameter count comes to **61.2%** of BERT-base's (66,955,010 vs 109,483,778, against a claimed 60%), and final validation accuracy reaches **95.3%** of BERT-base's own (83.8% vs 88.0%, against a claimed 97%). Training time favours DistilBERT by a wide margin too -- **279.5s** against BERT-base's **1,194.3s**, a **4.27x** speedup, well past the claimed 40% -- though that exact multiplier is directional, not precise: this run was timed on shared, non-dedicated CPU hardware, where wall-clock training time is sensitive to whatever else the machine happens to be doing at that moment, unlike a parameter count or an accuracy score, neither of which depends on the hardware running them at all. Common mistake: treating a single timing measurement with the same confidence as an accuracy measurement -- they are not the same kind of number. Practical takeaway: DistilBERT's size and accuracy trade-offs are what to trust from a run like this one; for a precise speed multiplier, you would want a dedicated, repeated benchmark, not a single notebook cell.

## Challenge -- Break the "Only the Head Changes" Assumption

**Problem.** A common assumption about fine-tuning is that it "just trains a tiny new head on top of frozen features" -- after all, the head is the only *newly initialised* piece. But Part 2's `Trainer` never froze anything: every one of BERT's own parameters was updated too. Let's break it deliberately: put that assumption to a real test and check, on real validation data, whether the encoder's own [CLS] representation actually moved -- using the exact probe Part 1 built, now run twice.

**Ingredients.** The 872 real validation sentences, with their true sentiment labels, run through two different encoders: the untouched pretrained encoder from Part 1, and Part 2's own fine-tuned BERT (its encoder half, ignoring the classification head).

In [11]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

def cls_embeddings(model, sentences, tok, batch_size=32):
    """[CLS] vector from the final hidden layer, for every sentence, batched."""
    model.eval()
    vecs = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]
            enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=64)
            out = model(**enc, output_hidden_states=True)
            vecs.append(out.hidden_states[-1][:, 0, :])  # [CLS] is always position 0
    return torch.cat(vecs, dim=0).numpy()

val_sentences = val_raw["sentence"]
val_labels = np.array(val_raw["label"])

cls_before = cls_embeddings(base_encoder, val_sentences, tokenizer)               # Part 1's untouched encoder
cls_after = cls_embeddings(bert_trainer.model, val_sentences, tokenizer)          # Part 2's fine-tuned model

print(f"CLS embeddings: before {cls_before.shape}, after {cls_after.shape}")

CLS embeddings: before (872, 768), after (872, 768)


**Idea -- a 2D projection, honestly.** Slide 10 built the head directly on the raw [CLS] vector, so that vector is the right thing to probe. Project each 768-dimensional set of vectors down to 2 dimensions with PCA -- a deterministic, linear projection, not t-SNE's stochastic, nonlinear one -- fit *separately* on each model's own embeddings, since the two encoders live in unrelated coordinate systems and only their own internal structure is being asked about. Colour every point by its true sentiment label, never shown to either model during this probe.

In [12]:
pca_before = PCA(n_components=2, random_state=0)
proj_before = pca_before.fit_transform(cls_before)

pca_after = PCA(n_components=2, random_state=0)
proj_after = pca_after.fit_transform(cls_after)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "BEFORE fine-tuning (Part 1's encoder)", "AFTER fine-tuning (Part 2's encoder)"))
for label, name, color in [(0, "negative", RED), (1, "positive", POS)]:
    mask = val_labels == label
    fig.add_trace(go.Scatter(x=proj_before[mask, 0], y=proj_before[mask, 1], mode="markers",
                              name=name, marker=dict(color=color, size=5, opacity=0.6),
                              legendgroup=name), row=1, col=1)
    fig.add_trace(go.Scatter(x=proj_after[mask, 0], y=proj_after[mask, 1], mode="markers",
                              name=name, marker=dict(color=color, size=5, opacity=0.6),
                              legendgroup=name, showlegend=False), row=1, col=2)
fig.update_layout(width=1050, height=480)
fig.show()

sil_before = silhouette_score(proj_before, val_labels)
sil_after = silhouette_score(proj_after, val_labels)

print(f"variance explained by 2 components -- before: {pca_before.explained_variance_ratio_.sum():.1%}, "
      f"after: {pca_after.explained_variance_ratio_.sum():.1%}")
print(f"silhouette score by true sentiment label (higher = better separated) -- "
      f"before: {sil_before:.3f}, after: {sil_after:.3f}")

variance explained by 2 components -- before: 16.5%, after: 65.2%
silhouette score by true sentiment label (higher = better separated) -- before: 0.015, after: 0.542


**Payoff.** Two numbers, and they both point the same way. First, just 2 principal components already capture **65.2%** of the fine-tuned encoder's total [CLS] variance, against only **16.5%** for the untouched pretrained encoder -- fine-tuning didn't add a little sentiment signal on the side, it concentrated a majority of the encoder's own variance into a low-dimensional axis a linear head can read off directly. Second, the silhouette score against the true sentiment label -- a standard measure of how cleanly two groups separate in a space, here run against labels the PCA projection itself never saw -- goes from **0.015** before fine-tuning to **0.542** after. The "just trains a tiny head" assumption does not survive this test: it broke, and the reason is exactly Part 2's own design choice -- nothing was frozen, so gradients reached every one of BERT's 109 million parameters, not only the 1,538 in the new classification head. The limitation is real too: 2 components is a small slice of 768 dimensions, so this probe is a lower bound on how separated the space really is, not the full picture. Practical takeaway: if you ever want to check whether a fine-tune actually changed a pretrained encoder or only rode on top of it, this before/after probe is cheap, and it does not require touching the training run at all -- only two forward passes and a projection.

## Reflection

**Problem.** You now own the full workflow: inspect a pretrained encoder, fine-tune it end to end, swap it for a smaller one and measure the real cost, and check what actually moved inside it. The point of this last cell is not a "correct" answer -- it is stating, in your own words, the reasoning each part was built on.

In [13]:
# 1. Part 1 measured cosine similarity between the [CLS] vectors of a
#    positive and a negative review, BEFORE any fine-tuning. What did that
#    number tell you about whether a pretrained encoder already "knows"
#    sentiment, versus knowing general language?
answer_1 = """
"""

# 2. Part 2 evaluated every epoch instead of only at the end. Looking at
#    your own three-point accuracy curve, was the model still improving by
#    epoch 3, or had it already flattened? How would you have answered that
#    question wrongly if you had only trained for 1 epoch?
answer_2 = """
"""

# 3. Part 3 changed exactly one variable (the encoder) and held everything
#    else fixed. Compare your own measured speedup and accuracy ratio to the
#    97% / 40%-faster claim from slide 12. Did your one small run land
#    close to that claim, or not -- and what about a 1,000-example, 3-epoch
#    run would make it land differently from the full published benchmark?
answer_3 = """
"""

# 4. The Challenge assumed fine-tuning "just trains a tiny head." Look at
#    your own before/after PCA scatter plots. Did the assumption survive?
#    Explain, in your own words, WHY updating every parameter (not just the
#    head) is what the plot's difference is actually evidence of.
answer_4 = """
"""

# 5. You now have three tools from this lab: a frozen-encoder probe (Part
#    1), an end-to-end fine-tune (Part 2), and a controlled model-swap
#    ablation (Part 3). If someone handed you a new text-classification
#    task tomorrow with only 500 labelled examples, which of these three
#    would you reach for first, and why?
answer_5 = """
"""


---
*BERT: Reading the Whole Sentence · Bluevision-ai Academy*